In [1]:
import plotly.express as px
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from datetime import datetime
import plotly.graph_objects as go
import numpy as np

In [2]:
df_themes = pd.read_csv("thematics_keywords_new.csv")

In [11]:
df_themes['Week'] = pd.to_datetime(df_themes['Week'])

weekly_crisis = (
    df_themes
    .groupby(['Week', 'Crise Type'], as_index=False)['Count']
    .sum()
)


legend_names = {
    'Crise climatique' : 'Climate Crisis',
    'Crise de la biodiversité' : 'Biodiverstity Crisis',
    'Crise des ressources' : 'Resource Crisis'
}

fig = go.Figure()

for crisis in weekly_crisis['Crise Type'].unique():
    subset = weekly_crisis[weekly_crisis['Crise Type'] == crisis]

    fig.add_trace(
        go.Scatter(
            x=subset['Week'],
            y=subset['Count'],
            mode='lines+markers',
            name=legend_names.get(crisis, crisis),
            line_shape='spline'
        )
    )

fig.update_layout(
    title='Weekly Keyword Counts by Crisis Type',
    xaxis_title='Week',
    yaxis_title='Total Keyword Count',
    hovermode='x unified',
    legend_title_text='Crisis Type'
)

fig.show()
fig.write_html("charts/Environmental_Crisis_Coverage.html", full_html=True, include_plotlyjs='cdn')

In [10]:
df_themes['Week'] = pd.to_datetime(df_themes['Week'])

sector_columns = [
    'General', 'Agriculture', 'Transport', 'Batiments',
    'Energie', 'Industrie', 'Eau', 'Ecosysteme', 'Economie Ressources'
]


weekly_sector = (
    df_themes.assign(
        **{
            sector: (
                df_themes[sector]
                .fillna(False)
                .astype(int)
                * df_themes['Count']
            )
            for sector in sector_columns
        }
    )
    .groupby('Week')[sector_columns]
    .sum()
    .reset_index()
)


weekly_sector_melted = weekly_sector.melt(
    id_vars='Week',
    value_vars=sector_columns,
    var_name='Sector',
    value_name='Count'
)


fig = go.Figure()

for sector in sector_columns:
    subset = weekly_sector_melted[
        weekly_sector_melted['Sector'] == sector
    ]

    fig.add_trace(
        go.Scatter(
            x=subset['Week'],
            y=subset['Count'],
            mode='lines',
            name=sector,
            line_shape='spline'
        )
    )

fig.update_layout(
    title='Sector Counts Over Time',
    xaxis_title='Week',
    yaxis_title='Count',
    hovermode='x unified',
    legend_title='Sector'
)

fig.show()
fig.write_html("charts/Sector_Counts_Over_Time.html", full_html=True, include_plotlyjs='cdn')

C:\Users\lucas\AppData\Local\Temp\ipykernel_24164\428750436.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


In [ ]:
# Sector columns
sector_columns = [
    
    'Agriculture',
    'Transport',
    'Batiments',
    'Energie',
    'Industrie',
    'Eau',
    'Ecosysteme',
    'Economie Ressources'
]


for sector in sector_columns:
    df_themes[sector] = df_themes[sector].fillna(False).astype(bool)


for sector in sector_columns:
    df_themes[sector] = df_themes[sector] * df_themes['Count']


sector_counts = (
    df_themes.groupby('Channel Title')[sector_columns]
      .sum()
      .reset_index()
)

# Convert to percentages 
channel_pct = sector_counts.copy()

channel_pct[sector_columns] = (
    channel_pct[sector_columns]
    .div(channel_pct[sector_columns].sum(axis=1), axis=0)
    * 100
)

# Pastel colors
sector_colors = {
    'Energie': '#F4A261',
    'Ecosysteme': '#B39DDB',
    'Industrie': '#F6D365',
    'Agriculture': '#7E83BC',
    'Economie Ressources': '#F4A6A6',
    'Transport': '#A8DADC',
    'Batiments': '#5B9BD5',
    'Eau': '#8BC34A',
    'General': '#D8BFD8'
}

# Plot
fig = go.Figure()

for sector in sector_columns:
    fig.add_trace(
        go.Bar(
            x=channel_pct['Channel Title'],
            y=channel_pct[sector],
            name=sector,
            marker_color=sector_colors.get(sector)
        )
    )

fig.update_layout(
    title='Sector Analysis per Channel - Flanders',
    barmode='stack',
    template='plotly_white',
    xaxis_title='Channel',
    yaxis_title='Sector Share (%)',
    legend_title='Sector',
    height=600
)

fig.update_yaxes(
    range=[0, 100],
    ticksuffix='%'
)

fig.show()
fig.write_html("charts/Sector_Analysis_per_Channel.html", full_html=True, include_plotlyjs='cdn')
